# Employee Attrition Prediction System
**Google Colab notebook | Reproducible multi-model classification**

This notebook implements the project brief: input validation, data-quality checks, repeatable preparation, multiple classifiers, evaluation, error review, visualisations, and exportable results.

> **Important:** No model can honestly guarantee accuracy above 90%. Attrition datasets vary, and class imbalance can make accuracy misleading. This notebook reports measured test-set results and also evaluates precision, recall, F1, ROC-AUC (when available), and a confusion matrix. Do not tune on the test set to force a target score. Use only public, synthetic, or explicitly authorised HR data; do not use predictions to make automated employment decisions.

**Dataset:** You can upload a CSV or use the IBM HR Analytics attrition sample from a public source. The notebook expects a binary target column such as `Attrition` with values `Yes/No`, `1/0`, or `True/False`. If using a different dataset, set the target column below.


## 1. Install / import libraries
Run cells from top to bottom. Colab usually already includes most of these packages.


In [ ]:
%pip -q install pandas numpy scikit-learn matplotlib seaborn joblib


In [ ]:
import io
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix,
    balanced_accuracy_score
)
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier, RandomForestClassifier, VotingClassifier, AdaBoostClassifier, RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.base import clone
import joblib

SEED = 42
TEST_SIZE = 0.20
TARGET_COLUMN = "Attrition"  # Change if your target column has a different name
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)
print("Libraries ready. Random seed:", SEED)


## 2. Load the dataset
Choose one method:
- **Upload your own CSV** (recommended if your project already has data).
- Or load the IBM HR Analytics sample CSV from a source you have permission to use.

The uploader cell works in Google Colab. If running outside Colab, use the local-path alternative below.


In [ ]:
# Google Colab upload
try:
    from google.colab import files
    uploaded = files.upload()
    csv_files = [name for name in uploaded if name.lower().endswith(".csv")]
    if not csv_files:
        raise ValueError("No CSV file uploaded. Upload a .csv dataset and rerun this cell.")
    DATA_PATH = csv_files[0]
    df_raw = pd.read_csv(io.BytesIO(uploaded[DATA_PATH]))
    print(f"Loaded {DATA_PATH!r}: {df_raw.shape[0]:,} rows × {df_raw.shape[1]:,} columns")
except ImportError:
    # Outside Colab: update this path to your CSV.
    DATA_PATH = "WA_Fn-UseC_-HR-Employee-Attrition.csv"
    df_raw = pd.read_csv(DATA_PATH)
    print(f"Loaded {DATA_PATH!r}: {df_raw.shape[0]:,} rows × {df_raw.shape[1]:,} columns")

display(df_raw.head())


## 3. Validate input and inspect data quality
This cell checks that the target exists, the file is non-empty, and reports missing values, duplicate rows, data types, and target distribution. Review any warnings before training.


In [ ]:
df = df_raw.copy()

if df.empty:
    raise ValueError("The uploaded dataset is empty.")
if TARGET_COLUMN not in df.columns:
    raise ValueError(
        f"Target column {TARGET_COLUMN!r} was not found. Available columns: {list(df.columns)}. "
        "Update TARGET_COLUMN and rerun from the imports onward."
    )

print("Shape:", df.shape)
print("\nColumn types:")
display(df.dtypes.astype(str).rename("dtype").to_frame())
print("\nMissing values (largest first):")
display(df.isna().sum().sort_values(ascending=False).rename("missing_count").to_frame())
print("\nDuplicate rows:", int(df.duplicated().sum()))
print("\nTarget counts:")
display(df[TARGET_COLUMN].value_counts(dropna=False).rename("count").to_frame())

if df[TARGET_COLUMN].isna().any():
    raise ValueError("Target column contains missing values. Remove or resolve these rows before modelling.")
if df[TARGET_COLUMN].nunique() < 2:
    raise ValueError("Target must contain at least two classes.")


## 4. Prepare target and features
- Drops exact duplicate rows.
- Removes rows with missing target values (already checked above).
- Excludes obvious identifier-like columns with unique values for nearly every row.
- Uses a stratified train/test split.
- Learns imputers, scaling, and one-hot encoding **on training data only** through a pipeline to avoid data leakage.

For the usual IBM HR sample, `Attrition` is the label. Review the automatically detected ID-like columns and override the list if needed.


In [ ]:
# Remove exact duplicates, preserving reproducibility.
df = df.drop_duplicates().reset_index(drop=True)

# Normalise common binary target encodings without assuming which class means attrition.
target_text = df[TARGET_COLUMN].astype(str).str.strip()
normal_map = {
    "yes": 1, "y": 1, "true": 1, "1": 1, "attrition": 1,
    "no": 0, "n": 0, "false": 0, "0": 0, "no attrition": 0
}
lower_values = set(target_text.str.lower().unique())
if lower_values.issubset(set(normal_map)):
    y = target_text.str.lower().map(normal_map).astype(int)
    target_mapping = {0: "No / negative class", 1: "Yes / positive class"}
else:
    # General binary target: encode the sorted class labels; inspect mapping before interpreting results.
    classes = sorted(target_text.unique())
    if len(classes) != 2:
        raise ValueError(f"Expected a binary target but found {len(classes)} classes: {classes}")
    mapping = {classes[0]: 0, classes[1]: 1}
    y = target_text.map(mapping).astype(int)
    target_mapping = {v: k for k, v in mapping.items()}

print("Target mapping (encoded label -> original meaning):", target_mapping)
print("Class balance:")
display(y.value_counts().sort_index().rename("count").to_frame())

X = df.drop(columns=[TARGET_COLUMN]).copy()

# Drop near-unique ID fields; review this list for your dataset.
id_like = [
    c for c in X.columns
    if X[c].nunique(dropna=True) >= max(20, int(0.98 * len(X)))
    and ("id" in c.lower() or c.lower() in {"employee number", "employeenumber", "record_id"})
]
print("Identifier-like columns excluded:", id_like if id_like else "None")
X = X.drop(columns=id_like, errors="ignore")

# Convert booleans to strings so they are handled consistently as categories.
for col in X.select_dtypes(include=["bool"]).columns:
    X[col] = X[col].astype(str)

# Detect numeric and categorical columns.
numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = [c for c in X.columns if c not in numeric_features]
print(f"Features: {X.shape[1]} total | {len(numeric_features)} numeric | {len(categorical_features)} categorical")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y
)
print("Train rows:", len(X_train), "| Test rows:", len(X_test))


## 5. Build preprocessing and model candidates
Models include a majority-class baseline plus Logistic Regression, Decision Tree, Random Forest, Extra Trees, Gradient Boosting, AdaBoost, SVM, KNN, and Naive Bayes. The same train/test split and preprocessing pipeline are used for fair comparison.

Some models can be slower on large datasets. The notebook is designed for typical student-sized HR data.


In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
], remainder="drop")

models = {
    "Dummy baseline": DummyClassifier(strategy="most_frequent"),
    "Logistic Regression": LogisticRegression(max_iter=3000, class_weight="balanced", random_state=SEED),
    "Decision Tree": DecisionTreeClassifier(class_weight="balanced", random_state=SEED, min_samples_leaf=3),
    "Random Forest": RandomForestClassifier(n_estimators=400, class_weight="balanced", random_state=SEED, n_jobs=-1, min_samples_leaf=2),
    "Extra Trees": ExtraTreesClassifier(n_estimators=400, class_weight="balanced", random_state=SEED, n_jobs=-1, min_samples_leaf=2),
    "Gradient Boosting": GradientBoostingClassifier(random_state=SEED),
    "AdaBoost": AdaBoostClassifier(random_state=SEED, n_estimators=150),
    "SVM (RBF)": SVC(C=1.0, kernel="rbf", probability=True, class_weight="balanced", random_state=SEED),
    "KNN": KNeighborsClassifier(n_neighbors=7),
    "Gaussian Naive Bayes": GaussianNB()
}
pipelines = {
    name: Pipeline(steps=[("preprocess", clone(preprocessor)), ("model", model)])
    for name, model in models.items()
}
print("Prepared", len(pipelines), "candidate models.")


## 6. Train and compare models
Metrics are calculated on the held-out test set. Accuracy is the fraction correct, while recall and F1 help reveal whether the model misses the minority attrition class. ROC-AUC is reported when probability scores are available and the test set contains both classes.


In [ ]:
results = []
fitted_models = {}

for name, pipe in pipelines.items():
    try:
        pipe.fit(X_train, y_train)
        pred = pipe.predict(X_test)
        row = {
            "Model": name,
            "Accuracy": accuracy_score(y_test, pred),
            "Balanced Accuracy": balanced_accuracy_score(y_test, pred),
            "Precision (positive class)": precision_score(y_test, pred, zero_division=0),
            "Recall (positive class)": recall_score(y_test, pred, zero_division=0),
            "F1 (positive class)": f1_score(y_test, pred, zero_division=0),
            "ROC-AUC": np.nan
        }
        if hasattr(pipe, "predict_proba") and y_test.nunique() == 2:
            scores = pipe.predict_proba(X_test)[:, 1]
            row["ROC-AUC"] = roc_auc_score(y_test, scores)
        elif hasattr(pipe, "decision_function") and y_test.nunique() == 2:
            row["ROC-AUC"] = roc_auc_score(y_test, pipe.decision_function(X_test))
        results.append(row)
        fitted_models[name] = pipe
        print(f"Finished: {name}")
    except Exception as exc:
        print(f"Skipped {name} due to error: {type(exc).__name__}: {exc}")

results_df = pd.DataFrame(results).sort_values(
    ["F1 (positive class)", "ROC-AUC", "Accuracy"], ascending=False, na_position="last"
).reset_index(drop=True)
display(results_df.style.format({
    "Accuracy": "{:.3f}", "Balanced Accuracy": "{:.3f}",
    "Precision (positive class)": "{:.3f}", "Recall (positive class)": "{:.3f}",
    "F1 (positive class)": "{:.3f}", "ROC-AUC": "{:.3f}"
}))

best_model_name = results_df.iloc[0]["Model"]
best_model = fitted_models[best_model_name]
print("Selected model by positive-class F1:", best_model_name)
print("Best model test accuracy:", f"{results_df.iloc[0]['Accuracy']:.2%}")
if results_df.iloc[0]["Accuracy"] < 0.90:
    print("Note: test accuracy is below 90%. Do not manipulate the test split to force a target; review data quality, features, class balance, and cross-validation instead.")


## 7. Visualise model comparison


In [ ]:
plot_df = results_df.sort_values("Accuracy", ascending=True)
plt.figure(figsize=(10, max(5, 0.45 * len(plot_df))))
sns.barplot(data=plot_df, x="Accuracy", y="Model")
plt.axvline(0.90, linestyle="--", label="90% reference (not a guarantee)")
plt.xlim(0, 1)
plt.title("Held-out test accuracy by model")
plt.xlabel("Accuracy")
plt.ylabel("")
plt.legend()
plt.tight_layout()
plt.show()

metric_cols = ["Precision (positive class)", "Recall (positive class)", "F1 (positive class)"]
results_df.set_index("Model")[metric_cols].plot(kind="bar", figsize=(12, 6), ylim=(0, 1))
plt.title("Positive-class metrics by model")
plt.ylabel("Score")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()


## 8. Inspect the selected model's errors
Check the confusion matrix and classification report. Confirm which encoded label represents attrition from the mapping in Section 4. False negatives (actual attrition predicted as no attrition) may matter differently from false positives, but this is an educational prototype—not a tool for employment decisions.


In [ ]:
best_pred = best_model.predict(X_test)
print("Selected model:", best_model_name)
print("\nClassification report:")
print(classification_report(y_test, best_pred, zero_division=0))

cm = confusion_matrix(y_test, best_pred, labels=[0, 1])
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Predicted 0", "Predicted 1"],
            yticklabels=["Actual 0", "Actual 1"])
plt.title(f"Confusion matrix — {best_model_name}")
plt.tight_layout()
plt.show()


## 9. Optional: stratified cross-validation on training data
Cross-validation provides a more stable estimate using only the training partition. It is not a replacement for the held-out test evaluation. This may take longer for several models.


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_rows = []
for name, pipe in pipelines.items():
    if name not in fitted_models:
        continue
    try:
        scores = cross_validate(
            pipe, X_train, y_train, cv=cv,
            scoring={"accuracy": "accuracy", "f1": "f1",
                     "precision": "precision", "recall": "recall"},
            n_jobs=None, error_score="raise"
        )
        cv_rows.append({
            "Model": name,
            "CV Accuracy mean": scores["test_accuracy"].mean(),
            "CV Accuracy std": scores["test_accuracy"].std(),
            "CV F1 mean": scores["test_f1"].mean(),
            "CV Precision mean": scores["test_precision"].mean(),
            "CV Recall mean": scores["test_recall"].mean()
        })
    except Exception as exc:
        print(f"CV skipped for {name}: {exc}")

cv_df = pd.DataFrame(cv_rows).sort_values("CV F1 mean", ascending=False)
display(cv_df.style.format({
    "CV Accuracy mean": "{:.3f}", "CV Accuracy std": "{:.3f}",
    "CV F1 mean": "{:.3f}", "CV Precision mean": "{:.3f}", "CV Recall mean": "{:.3f}"
}))


## 10. Feature importance / interpretability
For tree-based models, this shows encoded feature importances. One-hot encoding expands categorical columns, so the displayed names are transformed feature names rather than the original column names. Importance is associational, not causal.


In [ ]:
def show_feature_importance(pipe, top_n=20):
    model = pipe.named_steps["model"]
    prep = pipe.named_steps["preprocess"]
    if not hasattr(model, "feature_importances_"):
        print(f"{type(model).__name__} does not expose built-in feature_importances_.")
        return
    try:
        names = prep.get_feature_names_out()
        imp = pd.Series(model.feature_importances_, index=names).sort_values(ascending=False).head(top_n)
        display(imp.rename("importance").to_frame())
        imp.sort_values().plot(kind="barh", figsize=(9, max(4, top_n * 0.28)))
        plt.title(f"Top {len(imp)} encoded feature importances — {best_model_name}")
        plt.tight_layout()
        plt.show()
    except Exception as exc:
        print("Could not extract feature names:", exc)

show_feature_importance(best_model)


## 11. Save model and export results
The fitted pipeline includes preprocessing, so it can be reused with raw columns that match the training schema. Keep the model file private if it contains information derived from real employee data.


In [ ]:
OUTPUT_DIR = Path("attrition_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

model_path = OUTPUT_DIR / "employee_attrition_pipeline.joblib"
metrics_path = OUTPUT_DIR / "model_comparison.csv"
predictions_path = OUTPUT_DIR / "test_predictions.csv"

joblib.dump({
    "model": best_model,
    "model_name": best_model_name,
    "target_column": TARGET_COLUMN,
    "target_mapping": target_mapping,
    "feature_columns": X.columns.tolist(),
    "random_seed": SEED
}, model_path)

results_df.to_csv(metrics_path, index=False)
pred_export = X_test.copy()
pred_export["actual_encoded"] = y_test.values
pred_export["predicted_encoded"] = best_pred
pred_export.to_csv(predictions_path, index=False)

print("Saved files:")
for p in [model_path, metrics_path, predictions_path]:
    print(f" - {p} ({p.stat().st_size:,} bytes)")

try:
    from google.colab import files
    files.download(str(metrics_path))
    files.download(str(predictions_path))
    files.download(str(model_path))
except ImportError:
    print("Not running in Colab: files are saved in", OUTPUT_DIR.resolve())


## 12. Eight documented checks / test cases
Run this cell to verify important input and workflow assumptions. These are lightweight checks, not a replacement for a formal test suite.


In [ ]:
checks = []

def record_check(name, condition, detail=""):
    passed = bool(condition)
    checks.append({"Check": name, "Status": "PASS" if passed else "FAIL", "Details": detail})
    assert passed, f"Check failed: {name}. {detail}"

record_check("Dataset is non-empty", len(df) > 0, f"{len(df)} rows")
record_check("Target column exists", TARGET_COLUMN in df.columns, TARGET_COLUMN)
record_check("Target has two classes", y.nunique() == 2, f"{y.nunique()} classes")
record_check("No missing target values", not y.isna().any())
record_check("Train/test split is non-empty", len(X_train) > 0 and len(X_test) > 0)
record_check("Train/test feature columns match", list(X_train.columns) == list(X_test.columns))
record_check("Split preserves both classes in test", y_test.nunique() == 2, f"test classes: {sorted(y_test.unique())}")
record_check("Selected model predicts test rows", len(best_pred) == len(y_test))
record_check("Metrics are bounded", results_df["Accuracy"].between(0, 1).all())
record_check("Predictions contain valid encoded labels", set(np.unique(best_pred)).issubset(set(np.unique(y_train))))

checks_df = pd.DataFrame(checks)
display(checks_df)
print(f"{(checks_df['Status'] == 'PASS').sum()} / {len(checks_df)} checks passed.")


## 13. Try a single educational prediction
Edit the example values below to match the raw feature columns. This uses the selected model and is intended for demonstration only. Do not use individual predictions to make employment decisions.


In [ ]:
# Example: use the first held-out row as a schema-valid demo input.
# Replace with a hypothetical/authorised record as appropriate.
example = X_test.iloc[[0]].copy()
example_prediction = best_model.predict(example)[0]
print("Encoded prediction:", int(example_prediction))
print("Decoded meaning:", target_mapping.get(int(example_prediction), str(example_prediction)))
if hasattr(best_model, "predict_proba"):
    print("Model probability scores:", best_model.predict_proba(example)[0])
display(example)


## Notes for your project report
- State the exact dataset source, version, license, and whether it is synthetic/public/authorised.
- Report the actual held-out metrics from this run; do not claim 90%+ unless the measured result supports it.
- Include class balance, split method, random seed, confusion matrix, and cross-validation summary.
- Explain limitations: dataset shift, class imbalance, missing or biased labels, proxy variables, and potential unfairness.
- Employee attrition modelling is sensitive. Avoid using protected or proxy attributes to make decisions about individuals, and do not automate hiring, firing, promotion, or other employment decisions based on this prototype.
